# **How to Download and Process Data for our Skiing Threat Index**
This tutorial will show the steps to download and process data from the GFS model to calculate where there are ideal skiing conditions

# First Step
# Run the First Data Download from GFS Database

### We evaluated three seperate events for our project
- The first one was our initial weather event, a snow storm from 02/22/2026
- The second one was a "high index event" to test that our index actually worked and would produce high values for this high index event
- The second one was a "low index event" to test that our index actually worked and would produce low values for this low index event.

## Initial Event
Import functions into jupyter hub

In [1]:
from herbie import Herbie, FastHerbie
import pandas as pd, numpy as np
import xarray as xr
import dask

### First Line: 
##### Getting the data from the correct date and time, which was the halfway through the winter storm we chose
### Second Line:
##### Create Herbie object (H)
##### This will download and manage the data from GFS weather model data
- Saving to the " Data " folder within Jupyter Hub directory
- You can also change the model used, it just may change some of the further analysis depending on formatting differences between models
### Third Line: 
##### Lists all of the unfiltered data stored by the fast herbie function

In [2]:
run = pd.Timestamp("2026-02-22-00")
H = Herbie(run, model="gfs", fxx=12, save_dir='./data/', overwrite=True)
table = H.inventory()


✅ Found ┊ model=gfs ┊ product=pgrb2.0p25 ┊ 2026-Feb-22 00:00 UTC F12 ┊ GRIB2 @ aws ┊ IDX @ aws


#### Selecting the Weather Variables from the dataset by searching for:
- the names of the variables
- the atmospheric layer needed for the index

** Remember it will all be in a string !! **

Then setting those equal to variables for later use

In [8]:
ss1 = r"(:APCP:surface:0-|SNOD|:TCDC:entire atmosphere:\d+\s|:TMP:2 m above ground:)"
ss2 = r"(:VGRD:10 m above ground|:UGRD:10 m above ground)"
ss3 = "CSNOW:surface:.+ave"

## You will have to repeat all of these steps for all three ss# strings:
##### We are using ss3 for reference

### Create Fast Herbie object (H) to pull the specific data from ss3 object 

In [12]:
H = FastHerbie([run], model="gfs", fxx=np.arange(6,240,6).tolist(), save_dir='./data/', overwrite=True)

### Use .inventory function to read the data specified within ss3 string and store within a table

In [9]:
H.inventory(ss3)

,grib_message,start_byte,end_byte,range,reference_time,valid_time,variable,level,forecast_time,search_this,FILE
0,605,423401358,423431529.0,423401358-423431529,2026-02-22,2026-02-22 06:00:00,CSNOW,surface,0-6 hour ave fcst,:CSNOW:surface:0-6 hour ave fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
1,605,427202660,427232451.0,427202660-427232451,2026-02-22,2026-02-22 12:00:00,CSNOW,surface,6-12 hour ave fcst,:CSNOW:surface:6-12 hour ave fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
2,605,427509427,427536881.0,427509427-427536881,2026-02-22,2026-02-22 18:00:00,CSNOW,surface,12-18 hour ave fcst,:CSNOW:surface:12-18 hour ave fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
3,605,428792619,428821540.0,428792619-428821540,2026-02-22,2026-02-23 00:00:00,CSNOW,surface,18-24 hour ave fcst,:CSNOW:surface:18-24 hour ave fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
4,605,429174348,429203236.0,429174348-429203236,2026-02-22,2026-02-23 06:00:00,CSNOW,surface,24-30 hour ave fcst,:CSNOW:surface:24-30 hour ave fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
5,605,430406911,430436135.0,430406911-430436135,2026-02-22,2026-02-23 12:00:00,CSNOW,surface,30-36 hour ave fcst,:CSNOW:surface:30-36 hour ave fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
6,605,428578568,428607377.0,428578568-428607377,2026-02-22,2026-02-23 18:00:00,CSNOW,surface,36-42 hour ave fcst,:CSNOW:surface:36-42 hour ave fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
7,605,425874990,425904546.0,425874990-425904546,2026-02-22,2026-02-24 00:00:00,CSNOW,surface,42-48 hour ave fcst,:CSNOW:surface:42-48 hour ave fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
8,605,426832711,426862446.0,426832711-426862446,2026-02-22,2026-02-24 06:00:00,CSNOW,surface,48-54 hour ave fcst,:CSNOW:surface:48-54 hour ave fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
9,605,423331731,423363191.0,423331731-423363191,2026-02-22,2026-02-24 12:00:00,CSNOW,surface,54-60 hour ave fcst,:CSNOW:surface:54-60 hour ave fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....


## Download the data specified in ss3 and save to variable filepath

In [13]:
filepath = H.download(ss3)


## Create one array package with all of the ss3 data to use for data analysis
#### - Downloaded array shown below for example

In [64]:
ds1 = xr.open_mfdataset(filepath, combine='nested', concat_dim='valid_time', engine="cfgrib")
ds1

Ignoring index file '/courses/meteo473/sp26/473_sp26_group1/data/gfs/20260222/subset_92efbee5__gfs.t00z.pgrb2.0p25.f000.5b7b6.idx' older than GRIB file
Ignoring index file '/courses/meteo473/sp26/473_sp26_group1/data/gfs/20260222/subset_92125abe__gfs.t00z.pgrb2.0p25.f012.5b7b6.idx' older than GRIB file
Ignoring index file '/courses/meteo473/sp26/473_sp26_group1/data/gfs/20260222/subset_92605abe__gfs.t00z.pgrb2.0p25.f030.5b7b6.idx' older than GRIB file
Ignoring index file '/courses/meteo473/sp26/473_sp26_group1/data/gfs/20260222/subset_92ca5abe__gfs.t00z.pgrb2.0p25.f114.5b7b6.idx' older than GRIB file
Ignoring index file '/courses/meteo473/sp26/473_sp26_group1/data/gfs/20260222/subset_921c5abe__gfs.t00z.pgrb2.0p25.f048.5b7b6.idx' older than GRIB file
Ignoring index file '/courses/meteo473/sp26/473_sp26_group1/data/gfs/20260222/subset_92fb5abe__gfs.t00z.pgrb2.0p25.f108.5b7b6.idx' older than GRIB file
Ignoring index file '/courses/meteo473/sp26/473_sp26_group1/data/gfs/20260222/subset_927

<xarray.Dataset> Size: 664MB
Dimensions:            (valid_time: 40, latitude: 721, longitude: 1440)
Coordinates:
  * valid_time         (valid_time) datetime64[ns] 320B 2026-02-22 ... 2026-0...
  * latitude           (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude          (longitude) float64 12kB 0.0 0.25 0.5 ... 359.5 359.8
    time               datetime64[ns] 8B 2026-02-22
    step               (valid_time) timedelta64[ns] 320B 0 days 00:00:00 ... ...
    surface            float64 8B 0.0
    heightAboveGround  float64 8B 2.0
    atmosphere         float64 8B 0.0
Data variables:
    sde                (valid_time, latitude, longitude) float32 166MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    t2m                (valid_time, latitude, longitude) float32 166MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    tp                 (valid_time, latitude, longitude) float32 166MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    tcc                (valid_time, latitude, longitude) float32 166MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-03-23T17:15 GRIB to CDM+CF via cfgrib-0.9.1...

## Download the data specified in ss2 and save to variable filepath

In [65]:
filepath = H.download(ss2)


/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expre

## Create one array package with all of the ss2 data to use for data analysis
#### - Downloaded array shown below for example

In [66]:
ds2 = xr.open_mfdataset(filepath, combine='nested', concat_dim='valid_time', engine="cfgrib")
ds2

Ignoring index file '/courses/meteo473/sp26/473_sp26_group1/data/gfs/20260222/subset_92b288d0__gfs.t00z.pgrb2.0p25.f006.5b7b6.idx' older than GRIB file
Ignoring index file '/courses/meteo473/sp26/473_sp26_group1/data/gfs/20260222/subset_920e88d0__gfs.t00z.pgrb2.0p25.f036.5b7b6.idx' older than GRIB file
Ignoring index file '/courses/meteo473/sp26/473_sp26_group1/data/gfs/20260222/subset_92efaa7d__gfs.t00z.pgrb2.0p25.f000.5b7b6.idx' older than GRIB file
Ignoring index file '/courses/meteo473/sp26/473_sp26_group1/data/gfs/20260222/subset_920e88d0__gfs.t00z.pgrb2.0p25.f024.5b7b6.idx' older than GRIB file
Ignoring index file '/courses/meteo473/sp26/473_sp26_group1/data/gfs/20260222/subset_928988d0__gfs.t00z.pgrb2.0p25.f018.5b7b6.idx' older than GRIB file
Ignoring index file '/courses/meteo473/sp26/473_sp26_group1/data/gfs/20260222/subset_921c88d0__gfs.t00z.pgrb2.0p25.f048.5b7b6.idx' older than GRIB file
Ignoring index file '/courses/meteo473/sp26/473_sp26_group1/data/gfs/20260222/subset_92f

<xarray.Dataset> Size: 332MB
Dimensions:            (valid_time: 40, latitude: 721, longitude: 1440)
Coordinates:
  * valid_time         (valid_time) datetime64[ns] 320B 2026-02-22T06:00:00 ....
  * latitude           (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude          (longitude) float64 12kB 0.0 0.25 0.5 ... 359.5 359.8
    time               datetime64[ns] 8B 2026-02-22
    step               (valid_time) timedelta64[ns] 320B 0 days 06:00:00 ... ...
    heightAboveGround  float64 8B 10.0
Data variables:
    u10                (valid_time, latitude, longitude) float32 166MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    v10                (valid_time, latitude, longitude) float32 166MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-03-23T17:15 GRIB to CDM+CF via cfgrib-0.9.1...

## Create one array package with all of the ss2 data to use for data analysis
- Downloaded array shown below for example
- This ds5 will be used

In [14]:
ds5 = xr.open_mfdataset(filepath, combine='nested', concat_dim='valid_time', engine="cfgrib")
ds5

/tmp/ipykernel_3918382/3578487131.py:1: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  ds5 = xr.open_mfdataset(filepath, combine='nested', concat_dim='valid_time', engine="cfgrib")


<xarray.Dataset> Size: 162MB
Dimensions:     (valid_time: 39, latitude: 721, longitude: 1440)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 312B 2026-02-24 ... 2026-03-03T18...
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    time        datetime64[ns] 8B 2026-02-22
    step        (valid_time) timedelta64[ns] 312B 2 days 00:00:00 ... 9 days ...
    surface     float64 8B 0.0
Data variables:
    csnow       (valid_time, latitude, longitude) float32 162MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-04-08T16:50 GRIB to CDM+CF via cfgrib-0.9.1...

## Merging the variables within the ss1 and ss2 dataset into one dataset called ds3

In [67]:
ds3 = xr.merge([ds1,ds2],compat='override')
ds3

/tmp/ipykernel_1099611/3661606351.py:1: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'valid_time' ('valid_time',) The recommendation is to set join explicitly for this case.
  ds3 = xr.merge([ds1,ds2],compat='override')


<xarray.Dataset> Size: 997MB
Dimensions:            (latitude: 721, longitude: 1440, valid_time: 40)
Coordinates:
  * latitude           (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude          (longitude) float64 12kB 0.0 0.25 0.5 ... 359.5 359.8
  * valid_time         (valid_time) datetime64[ns] 320B 2026-02-22 ... 2026-0...
    time               datetime64[ns] 8B 2026-02-22
    step               (valid_time) timedelta64[ns] 320B 0 days 00:00:00 ... ...
    surface            float64 8B 0.0
    heightAboveGround  float64 8B 2.0
    atmosphere         float64 8B 0.0
Data variables:
    sde                (valid_time, latitude, longitude) float32 166MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    t2m                (valid_time, latitude, longitude) float32 166MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    tp                 (valid_time, latitude, longitude) float32 166MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    tcc                (valid_time, latitude, longitude) float32 166MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    u10                (valid_time, latitude, longitude) float32 166MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    v10                (valid_time, latitude, longitude) float32 166MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-03-23T17:15 GRIB to CDM+CF via cfgrib-0.9.1...

## Getting area specific data only

##### Here we focused on the Northeast region of the US, so if you wish to focus on a different area change the latitude and longitude splices as needed


In [15]:
ds6 = ds5.sel(latitude=slice(55,35), longitude=slice(280,300))


## Saving dataset to a netcdf file
##### fname is selecting the name for the netcdf file using datetime functions. 

##### This will automatically format the name to match the actual datetime of the model run.

In [69]:
fname = f'gfs_{run:%Y%m%d%H}.nc'
ds4.to_netcdf(fname)

## Saving dataset to netcdf file

In [16]:
ds = xr.open_dataset('gfs_2026022200.nc')

sh: 1: getfattr: not found


## Merge ds and ds6 for final dataset (ds7)

In [18]:
ds7 = xr.merge([ds, ds6])

/tmp/ipykernel_3918382/226933052.py:1: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'valid_time' ('valid_time',) The recommendation is to set join explicitly for this case.
  ds7 = xr.merge([ds, ds6])
/tmp/ipykernel_3918382/226933052.py:1: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds7 = xr.merge([ds, ds6])
/tmp/ipykernel_3918382/226933052.py:1: FutureWarning: In a future version of xarray the default value for compat w

## Adding ds7 to saved netcdf file

In [19]:
ds7.to_netcdf('gfs.nc')

# Second Step

## Second high end event

### Getting the data from the correct date and time, which was the halfway through the winter storm we chose

In [2]:
run = pd.Timestamp("2026-01-25-00")

#### Selecting the Weather Variables from the dataset by searching for:
- the names of the variables
- the atmospheric layer needed for the index

** Remember it will all be in a string !! **

Then setting those equal to variables for later use

In [3]:
ss1 = r"(:APCP:surface:0-|SNOD|:TCDC:entire atmosphere:\d+\s|:TMP:2 m above ground:)"
ss2 = r"(:VGRD:10 m above ground|:UGRD:10 m above ground)"
ss3 = "CSNOW:surface:.+ave"

## Create Fast Herbie object (H)
### This will download and manage the data from GFS weather model data
- Saving to the " Data " folder within Jupyter Hub directory

In [4]:
H = FastHerbie([run], model="gfs", fxx=np.arange(6,240,6).tolist(), save_dir='./data/', overwrite=True)

## Use .inventory function to read the data specified within ss1 string and store within a table

In [5]:
H.inventory(ss1)

/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expre

,grib_message,start_byte,end_byte,range,reference_time,valid_time,variable,level,forecast_time,search_this,FILE
0,578,408693462,409225447.0,408693462-409225447,2026-01-25,2026-01-25 06:00:00,SNOD,surface,6 hour fcst,:SNOD:surface:6 hour fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
1,581,409847076,410719835.0,409847076-410719835,2026-01-25,2026-01-25 06:00:00,TMP,2 m above ground,6 hour fcst,:TMP:2 m above ground:6 hour fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
2,596,421366264,421717803.0,421366264-421717803,2026-01-25,2026-01-25 06:00:00,APCP,surface,0-6 hour acc fcst,:APCP:surface:0-6 hour acc fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
3,597,421717804,422069343.0,421717804-422069343,2026-01-25,2026-01-25 06:00:00,APCP,surface,0-6 hour acc fcst,:APCP:surface:0-6 hour acc fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
4,636,439065693,439903669.0,439065693-439903669,2026-01-25,2026-01-25 06:00:00,TCDC,entire atmosphere,6 hour fcst,:TCDC:entire atmosphere:6 hour fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
...,...,...,...,...,...,...,...,...,...,...,...
152,636,449512551,450353708.0,449512551-450353708,2026-01-25,2026-02-03 12:00:00,TCDC,entire atmosphere,228 hour fcst,:TCDC:entire atmosphere:228 hour fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
153,578,416816898,417364390.0,416816898-417364390,2026-01-25,2026-02-03 18:00:00,SNOD,surface,234 hour fcst,:SNOD:surface:234 hour fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
154,581,417984514,418872716.0,417984514-418872716,2026-01-25,2026-02-03 18:00:00,TMP,2 m above ground,234 hour fcst,:TMP:2 m above ground:234 hour fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
155,597,429245715,430055292.0,429245715-430055292,2026-01-25,2026-02-03 18:00:00,APCP,surface,0-234 hour acc fcst,:APCP:surface:0-234 hour acc fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....


## Download the data specified in ss1 and save to variable filepath

In [6]:
filepath = H.download(ss1)

/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expre

## Create one array package with all of the ss1 data to use for data analysis
#### - Downloaded array shown below for example

In [7]:
ds1 = xr.open_mfdataset(filepath, combine='nested', concat_dim='valid_time', engine="cfgrib")
ds1

/tmp/ipykernel_613174/757839623.py:1: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  ds1 = xr.open_mfdataset(filepath, combine='nested', concat_dim='valid_time', engine="cfgrib")


<xarray.Dataset> Size: 648MB
Dimensions:            (valid_time: 39, latitude: 721, longitude: 1440)
Coordinates:
  * valid_time         (valid_time) datetime64[ns] 312B 2026-01-29 ... 2026-0...
  * latitude           (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude          (longitude) float64 12kB 0.0 0.25 0.5 ... 359.5 359.8
    time               datetime64[ns] 8B 2026-01-25
    step               (valid_time) timedelta64[ns] 312B 4 days 00:00:00 ... ...
    surface            float64 8B 0.0
    heightAboveGround  float64 8B 2.0
    atmosphere         float64 8B 0.0
Data variables:
    sde                (valid_time, latitude, longitude) float32 162MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    t2m                (valid_time, latitude, longitude) float32 162MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    tp                 (valid_time, latitude, longitude) float32 162MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    tcc                (valid_time, latitude, longitude) float32 162MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-04-13T17:24 GRIB to CDM+CF via cfgrib-0.9.1...

## Repeat the process again for ss2

In [8]:
filepath = H.download(ss2)

/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expre

In [9]:
ds2 = xr.open_mfdataset(filepath, combine='nested', concat_dim='valid_time', engine="cfgrib")
ds2

/tmp/ipykernel_613174/485652326.py:1: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  ds2 = xr.open_mfdataset(filepath, combine='nested', concat_dim='valid_time', engine="cfgrib")


<xarray.Dataset> Size: 324MB
Dimensions:            (valid_time: 39, latitude: 721, longitude: 1440)
Coordinates:
  * valid_time         (valid_time) datetime64[ns] 312B 2026-01-25T06:00:00 ....
  * latitude           (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude          (longitude) float64 12kB 0.0 0.25 0.5 ... 359.5 359.8
    time               datetime64[ns] 8B 2026-01-25
    step               (valid_time) timedelta64[ns] 312B 0 days 06:00:00 ... ...
    heightAboveGround  float64 8B 10.0
Data variables:
    u10                (valid_time, latitude, longitude) float32 162MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    v10                (valid_time, latitude, longitude) float32 162MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-04-13T17:24 GRIB to CDM+CF via cfgrib-0.9.1...

## Repeat the process again for ss3

In [10]:
filepath = H.download(ss3)

In [11]:
ds3 = xr.open_mfdataset(filepath, combine='nested', concat_dim='valid_time', engine="cfgrib")
ds3


/tmp/ipykernel_613174/3447206367.py:1: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  ds3 = xr.open_mfdataset(filepath, combine='nested', concat_dim='valid_time', engine="cfgrib")


<xarray.Dataset> Size: 162MB
Dimensions:     (valid_time: 39, latitude: 721, longitude: 1440)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 312B 2026-01-25T18:00:00 ... 2026...
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    time        datetime64[ns] 8B 2026-01-25
    step        (valid_time) timedelta64[ns] 312B 0 days 18:00:00 ... 8 days ...
    surface     float64 8B 0.0
Data variables:
    csnow       (valid_time, latitude, longitude) float32 162MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-04-13T17:24 GRIB to CDM+CF via cfgrib-0.9.1...

## Merging the three arrays into one dataset
#### Merge the three dataset names using xarray 'merge' function

In [12]:
ds4 = xr.merge([ds1,ds2,ds3],compat='override')
ds4

/tmp/ipykernel_613174/3586962655.py:1: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'valid_time' ('valid_time',) The recommendation is to set join explicitly for this case.
  ds4 = xr.merge([ds1,ds2,ds3],compat='override')


<xarray.Dataset> Size: 1GB
Dimensions:            (latitude: 721, longitude: 1440, valid_time: 39)
Coordinates:
  * latitude           (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude          (longitude) float64 12kB 0.0 0.25 0.5 ... 359.5 359.8
  * valid_time         (valid_time) datetime64[ns] 312B 2026-01-25T06:00:00 ....
    time               datetime64[ns] 8B 2026-01-25
    step               (valid_time) timedelta64[ns] 312B 0 days 06:00:00 ... ...
    surface            float64 8B 0.0
    heightAboveGround  float64 8B 2.0
    atmosphere         float64 8B 0.0
Data variables:
    sde                (valid_time, latitude, longitude) float32 162MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    t2m                (valid_time, latitude, longitude) float32 162MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    tp                 (valid_time, latitude, longitude) float32 162MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    tcc                (valid_time, latitude, longitude) float32 162MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    u10                (valid_time, latitude, longitude) float32 162MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    v10                (valid_time, latitude, longitude) float32 162MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    csnow              (valid_time, latitude, longitude) float32 162MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-04-13T17:24 GRIB to CDM+CF via cfgrib-0.9.1...

## Getting area specific data only
##### Here we focused on the Northeast region of the US, so if you wish to focus on a different area change the latitude and longitude splices as needed

In [13]:
ds5 = ds4.sel(latitude=slice(55,35), longitude=slice(280,300))


## Saving dataset to a netcdf file
##### fname is selecting the name for the netcdf file using datetime functions. 

##### This will automatically format the name to match the actual datetime of the model run.

In [14]:
fname = f'gfs_{run:%Y%m%d%H}.nc'
ds5.to_netcdf(fname)

## Saving dataset to netcdf file

In [15]:
ds = xr.open_dataset('gfs_2026012500.nc')

sh: 1: getfattr: not found


## Adding ds5 to netcdf file

In [18]:
ds5.to_netcdf('gfs.nc_high')

sh: 1: getfattr: not found


# Repeat above steps for section 2 "High Index Event" for chosen "Low event index"